# Class-Agnostic RPN Training on Roboflow YOLO Dataset (Boxes Only)

In [ ]:
!pip install torch torchvision roboflow --quiet

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F

## Dataset Loader (YOLO format → Boxes only, ignore classes)

In [ ]:
class YOLOBBoxDataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png','.jpeg'))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.rsplit('.',1)[0] + '.txt')

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.strip().split())
                    # YOLO -> xyxy
                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h
                    boxes.append([x1, y1, x2, y2])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)  # class-agnostic = object=1

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        img = F.to_tensor(img)

        return img, target

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

## Paths (Roboflow dataset structure)

In [ ]:
train_img = '../train/images'
train_lbl = '../train/labels'

val_img = '../valid/images'
val_lbl = '../valid/labels'

test_img = '../test/images'
test_lbl = '../test/labels'

In [ ]:
train_ds = YOLOBBoxDataset(train_img, train_lbl)
val_ds = YOLOBBoxDataset(val_img, val_lbl)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

## Model: Faster R-CNN (class-agnostic = 2 classes only)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')

# Replace head for class-agnostic detection (background + object)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 2)

model.to(device)

## Training Loop (RPN + ROI heads trained implicitly)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 5

model.train()

for epoch in range(num_epochs):
    epoch_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}")

## Inference Example

In [ ]:
model.eval()

img, _ = train_ds[0]
with torch.no_grad():
    pred = model([img.to(device)])

print(pred[0]['boxes'][:5])